In [0]:
catalog = "retail"
schema = "sales"
volume = "autoloader_demo"

base = f"/Volumes/{catalog}/{schema}/{volume}/auto_loader"

landing = f"{base}/landing"
schema_loc = f"{base}/_schema"
checkpoint = f"{base}/_checkpoint"

table_name = f"{catalog}.{schema}.orders_bronze"

#_checkpoint

# belongs to  Auto Loader ingestion query.


In [0]:
orders_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_loc)
    .load(landing)
)

In [0]:
orders_stream.printSchema()

root
 |-- amount: string (nullable = true)
 |-- customer: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- _rescued_data: string (nullable = true)



In [0]:
from pyspark.sql.functions import col, current_timestamp

orders_with_time = (orders_stream.withColumn("event_time", current_timestamp()))

In [0]:
orders_with_time.printSchema()

root
 |-- amount: string (nullable = true)
 |-- customer: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- event_time: timestamp (nullable = false)



In [0]:
display(orders_with_time.show()) #definition of an ongoing stream, not a static batch of rows sitting there ready to display

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-4600301662461843>, line 1
----> 1 display(orders_with_time.show())

File <command-7358200838579043>, line 3
      1 from pyspark.sql.functions import col, current_timestamp
----> 3 orders_with_time = (orders_stream.withColumn("event_time", current_timestamp()))

File <command-7358200838579039>, line 2
      1 orders_stream = (
----> 2     spark.readStream
      3     .format("cloudFiles")
      4     .option("cloudFiles.format", "json")
      5     .option("cloudFiles.schemaLocation", schema_loc)
      6     .load(landing)
      7 )

AnalysisException: Queries with streaming sources must be executed with writeStream.start(), or from a streaming table or flow definition within a Lakeflow Declarative Pipeline.;
cloudFiles

JVM stacktrace:
org.apache.spark.sql.catalyst.ExtendedAnalysisException
	at org.apache.spark.sql.cataly

In [0]:


#tumbling window
from pyspark.sql.functions import window, col

windowed_orders = (orders_with_time.withWatermark("event_time", "10 minutes").groupBy(window(col("event_time"), "5 minutes"),col("customer"))
    .count())

Belongs to this window + watermark aggregation query.

In [0]:
practice_checkpoint = f"{base}/_checkpoint_watermark_practice"

# Belongs to this window + watermark aggregation query.

In [0]:
query = (
    windowed_orders
    .writeStream
    .format("memory")
    .outputMode("update")
    .option("checkpointLocation", practice_checkpoint)
    .queryName("window_test")
    .trigger(availableNow=True)
    .start()
)



In [0]:
query.awaitTermination()

In [0]:
spark.sql("SELECT * FROM window_test ORDER BY window.start").show(truncate=False)

+------------------------------------------+--------+-----+
|window                                    |customer|count|
+------------------------------------------+--------+-----+
|{2026-10-02 09:15:00, 2026-10-02 09:20:00}|Rahul   |1    |
|{2026-10-02 09:15:00, 2026-10-02 09:20:00}|Sanjay  |1    |
|{2026-10-02 09:15:00, 2026-10-02 09:20:00}|Meera   |1    |
|{2026-10-02 09:15:00, 2026-10-02 09:20:00}|Aditi   |1    |
+------------------------------------------+--------+-----+



In [0]:
''' [
  {"customer": "Rahul", "event_time": "2026-10-02 10:00:00"},
  {"customer": "Rahul", "event_time": "2026-10-02 10:07:00"},
  {"customer": "Rahul", "event_time": "2026-10-02 10:20:00"}
] '''

In [0]:
from pyspark.sql import Row
from datetime import datetime

test_data = [
    Row(customer="Rahul", event_time=datetime(2026, 10, 2, 10, 0)),
    Row(customer="Rahul", event_time=datetime(2026, 10, 2, 10, 7)),
    Row(customer="Rahul", event_time=datetime(2026, 10, 2, 10, 20))
]

test_df = spark.createDataFrame(test_data)

test_df.show()


+--------+-------------------+
|customer|         event_time|
+--------+-------------------+
|   Rahul|2026-10-02 10:00:00|
|   Rahul|2026-10-02 10:07:00|
|   Rahul|2026-10-02 10:20:00|
+--------+-------------------+



In [0]:
#2. Apply window + watermark


test_windowed = (test_df .withWatermark("event_time", "10 minutes").groupBy(window(col("event_time"), "5 minutes"), col("customer")  )
    .count()
)

In [0]:
#3. Easiest proper experiment: create JSON files

#Create JSON file 1:

{"customer":"Rahul","event_time":"2026-10-02T10:00:00"}
{"customer":"Rahul","event_time":"2026-10-02T10:07:00"}



{'customer': 'Rahul', 'event_time': '2026-10-02T10:07:00'}

In [0]:
#Then file 2 later:

{"customer":"Rahul","event_time":"2026-10-02T10:20:00"}



{'customer': 'Rahul', 'event_time': '2026-10-02T10:20:00'}

In [0]:
#Then file 3 as the late event:

{"customer":"Rahul","event_time":"2026-10-02T10:03:00"}

{'customer': 'Rahul', 'event_time': '2026-10-02T10:03:00'}

In [0]:
#4. Read those files with Auto Loader
test_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .schema("customer string, event_time timestamp")
    .load(landing)
)

In [0]:
#5. Window + watermark
test_windowed = (
    test_stream
    .withWatermark("event_time", "10 minutes")
    .groupBy(window(col("event_time"), "5 minutes"),col("customer"))
    .count()
)

In [0]:
#6. Run with your cluster's AvailableNow

#Use a new checkpoint:

late_checkpoint1 = f"{base}/_checkpoint_late_event1"

query = (
    test_windowed
    .writeStream
    .format("memory")
    .outputMode("update")
    .option("checkpointLocation", late_checkpoint1)
    .queryName("late_event_test")
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()

In [0]:
spark.sql("SELECT * FROM late_event_test ORDER BY window.start ").show(truncate=False)

+------+--------+-----+
|window|customer|count|
+------+--------+-----+
+------+--------+-----+



In [0]:
display(dbutils.fs.ls(landing))

path,name,size,modificationTime
dbfs:/Volumes/retail/sales/autoloader_demo/auto_loader/landing/orders_01.json,orders_01.json,104,1790506673000
dbfs:/Volumes/retail/sales/autoloader_demo/auto_loader/landing/orders_02.json,orders_02.json,52,1790506673000
dbfs:/Volumes/retail/sales/autoloader_demo/auto_loader/landing/orders_03.json,orders_03.json,53,1790506783000


In [0]:
practice_landing = f"{base}/watermark_practice/landing"
practice_checkpoint = f"{base}/watermark_practice/checkpoint"

In [0]:
{"customer":"Rahul","event_time":"2026-10-02T10:00:00"}
{"customer":"Rahul","event_time":"2026-10-02T10:07:00"}

{'customer': 'Rahul', 'event_time': '2026-10-02T10:07:00'}

In [0]:
json_text = """{"customer":"Rahul","event_time":"2026-10-02T10:00:00"}
{"customer":"Rahul","event_time":"2026-10-02T10:07:00"}"""

dbutils.fs.mkdirs(practice_landing)

dbutils.fs.put(
    f"{practice_landing}/batch1.json",
    json_text,
    True
)

Wrote 111 bytes.


True

In [0]:
display(dbutils.fs.ls(practice_landing))

path,name,size,modificationTime
dbfs:/Volumes/retail/sales/autoloader_demo/auto_loader/watermark_practice/landing/batch1.json,batch1.json,111,1790934426000


In [0]:
test_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .schema("customer string, event_time timestamp")
    .load(practice_landing)
)

In [0]:
test_windowed = (
    test_stream
    .withWatermark("event_time", "10 minutes")
    .groupBy(
        window(col("event_time"), "5 minutes"),
        col("customer")
    )
    .count()
)

In [0]:
query = (
    test_windowed
    .writeStream
    .format("memory")
    .outputMode("update")
    .option("checkpointLocation", practice_checkpoint)
    .queryName("watermark_practice")
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()

In [0]:
spark.sql("SELECT * FROM watermark_practice ORDER BY window.start").show(truncate=False)

+------------------------------------------+--------+-----+
|window                                    |customer|count|
+------------------------------------------+--------+-----+
|{2026-10-02 10:00:00, 2026-10-02 10:05:00}|Rahul   |1    |
|{2026-10-02 10:05:00, 2026-10-02 10:10:00}|Rahul   |1    |
+------------------------------------------+--------+-----+



In [0]:
dbutils.fs.put(
    f"{practice_landing}/batch2.json",
    '{"customer":"Rahul","event_time":"2026-10-02T10:20:00"}',
    True
)

Wrote 55 bytes.


True

In [0]:
display(dbutils.fs.ls(practice_landing))

path,name,size,modificationTime
dbfs:/Volumes/retail/sales/autoloader_demo/auto_loader/watermark_practice/landing/batch1.json,batch1.json,111,1790934426000
dbfs:/Volumes/retail/sales/autoloader_demo/auto_loader/watermark_practice/landing/batch2.json,batch2.json,55,1790934604000


In [0]:
'''Step 5 — Create a NEW checkpoint and Delta output

Because the previous checkpoint belongs to the memory-sink query, use a fresh checkpoint:'''

In [0]:
delta_checkpoint = f"{base}/watermark_practice/delta_checkpoint"
delta_output = f"{base}/watermark_practice/output"

In [0]:
'''Step 6 — Start the same streaming aggregation with Delta

 test_stream and test_windowed remain exactly as they are.'''

query = (
    test_windowed
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", delta_checkpoint)
    .trigger(availableNow=True)
    .start(delta_output)
)

query.awaitTermination()

In [0]:
result = spark.read.format("delta").load(delta_output)

result.orderBy("window.start").show(truncate=False)

+------------------------------------------+--------+-----+
|window                                    |customer|count|
+------------------------------------------+--------+-----+
|{2026-10-02 10:00:00, 2026-10-02 10:05:00}|Rahul   |1    |
|{2026-10-02 10:05:00, 2026-10-02 10:10:00}|Rahul   |1    |
+------------------------------------------+--------+-----+



In [0]:
dbutils.fs.put(
    f"{practice_landing}/batch3.json",
    '{"customer":"Rahul","event_time":"2026-10-02T10:03:00"}',
    True
)

Wrote 55 bytes.


True

In [0]:
query = (
    test_windowed
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", delta_checkpoint)
    .trigger(availableNow=True)
    .start(delta_output)
)

query.awaitTermination()

In [0]:
result = spark.read.format("delta").load(delta_output)

result.orderBy("window.start").show(truncate=False)

+------------------------------------------+--------+-----+
|window                                    |customer|count|
+------------------------------------------+--------+-----+
|{2026-10-02 10:00:00, 2026-10-02 10:05:00}|Rahul   |1    |
|{2026-10-02 10:05:00, 2026-10-02 10:10:00}|Rahul   |1    |
+------------------------------------------+--------+-----+



10:00–10:05   Rahul   1
10:05–10:10   Rahul   1

Even though we later added:

10:03   Rahul

the first window did not become 2.

What happened?

The event sequence was:

10:00  → window 10:00–10:05
10:07  → window 10:05–10:10
10:20  → latest event

When Spark saw 10:20:

watermark = 10:20 - 10 minutes
          = 10:10

So:

Watermark = 10:10

Then we sent:

10:03   ← late event

But:

10:03 < 10:10

Therefore it was too late for the stateful window aggregation and did not update the already-finalized 10:00–10:05 result.

The key thing we should remember

Watermark is not a timer.

It is an event-time cutoff.

Latest event time
       ↓
    10:20
       ↓
minus allowed lateness (10 min)
       ↓
   watermark
    10:10
       ↓
older state/windows can be finalized/cleaned

And that's why the 10:03 event didn't increase the count.

we have now actually demonstrated tumbling window + watermark + late event in Databricks, rather than just reading the theory. 🎯

In [0]:
# Now let's add one more event to prove the difference between a late-but-allowed event and a too-late event.

# Step 11 — Add an event at 10:17

In [0]:
dbutils.fs.put(
    f"{practice_landing}/batch4.json",
    '{"customer":"Rahul","event_time":"2026-10-02T10:17:00"}',
    True
)

Wrote 55 bytes.


True

In [0]:
display(dbutils.fs.ls(practice_landing))

path,name,size,modificationTime
dbfs:/Volumes/retail/sales/autoloader_demo/auto_loader/watermark_practice/landing/batch1.json,batch1.json,111,1790934426000
dbfs:/Volumes/retail/sales/autoloader_demo/auto_loader/watermark_practice/landing/batch2.json,batch2.json,55,1790934604000
dbfs:/Volumes/retail/sales/autoloader_demo/auto_loader/watermark_practice/landing/batch3.json,batch3.json,55,1790934890000
dbfs:/Volumes/retail/sales/autoloader_demo/auto_loader/watermark_practice/landing/batch4.json,batch4.json,55,1790935058000


In [0]:
query = (
    test_windowed
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", delta_checkpoint)
    .trigger(availableNow=True)
    .start(delta_output)
)

query.awaitTermination()

In [0]:
result = spark.read.format("delta").load(delta_output)

result.orderBy("window.start").show(truncate=False)

+------------------------------------------+--------+-----+
|window                                    |customer|count|
+------------------------------------------+--------+-----+
|{2026-10-02 10:00:00, 2026-10-02 10:05:00}|Rahul   |1    |
|{2026-10-02 10:05:00, 2026-10-02 10:10:00}|Rahul   |1    |
+------------------------------------------+--------+-----+



In [0]:
dbutils.fs.put(
    f"{practice_landing}/batch5.json",
    '{"customer":"Rahul","event_time":"2026-10-02T10:18:00"}',
    True
)

Wrote 55 bytes.


True

In [0]:
query = (
    test_windowed
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", delta_checkpoint)
    .trigger(availableNow=True)
    .start(delta_output)
)

query.awaitTermination()

In [0]:
result = spark.read.format("delta").load(delta_output)

result.orderBy("window.start").show(truncate=False)

+------------------------------------------+--------+-----+
|window                                    |customer|count|
+------------------------------------------+--------+-----+
|{2026-10-02 10:00:00, 2026-10-02 10:05:00}|Rahul   |1    |
|{2026-10-02 10:05:00, 2026-10-02 10:10:00}|Rahul   |1    |
|{2026-10-02 10:15:00, 2026-10-02 10:20:00}|Rahul   |1    |
+------------------------------------------+--------+-----+



In [0]:
dbutils.fs.put(
    f"{practice_landing}/batch6.json",
    '{"customer":"Rahul","event_time":"2026-10-02T10:30:00"}',
    True
)

Wrote 55 bytes.


True

In [0]:
query = (
    test_windowed
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", delta_checkpoint)
    .trigger(availableNow=True)
    .start(delta_output)
)

query.awaitTermination()

In [0]:
result = spark.read.format("delta").load(delta_output)

result.orderBy("window.start").show(truncate=False)

+------------------------------------------+--------+-----+
|window                                    |customer|count|
+------------------------------------------+--------+-----+
|{2026-10-02 10:00:00, 2026-10-02 10:05:00}|Rahul   |1    |
|{2026-10-02 10:05:00, 2026-10-02 10:10:00}|Rahul   |1    |
|{2026-10-02 10:15:00, 2026-10-02 10:20:00}|Rahul   |1    |
+------------------------------------------+--------+-----+



In [0]:
dbutils.fs.put(
    f"{practice_landing}/batch7.json",
    '{"customer":"Rahul","event_time":"2026-10-02T10:35:00"}',
    True
)

Wrote 55 bytes.


True

In [0]:
query = (
    test_windowed
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", delta_checkpoint)
    .trigger(availableNow=True)
    .start(delta_output)
)

query.awaitTermination()

In [0]:
result = spark.read.format("delta").load(delta_output)

result.orderBy("window.start").show(truncate=False)

+------------------------------------------+--------+-----+
|window                                    |customer|count|
+------------------------------------------+--------+-----+
|{2026-10-02 10:00:00, 2026-10-02 10:05:00}|Rahul   |1    |
|{2026-10-02 10:05:00, 2026-10-02 10:10:00}|Rahul   |1    |
|{2026-10-02 10:15:00, 2026-10-02 10:20:00}|Rahul   |1    |
|{2026-10-02 10:20:00, 2026-10-02 10:25:00}|Rahul   |1    |
+------------------------------------------+--------+-----+



In [0]:
dbutils.fs.put(
    f"{practice_landing}/batch8.json",
    '{"customer":"Rahul","event_time":"2026-10-02T10:23:00"}',
    True
)

Wrote 55 bytes.


True

In [0]:
query = (
    test_windowed
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", delta_checkpoint)
    .trigger(availableNow=True)
    .start(delta_output)
)

query.awaitTermination()

In [0]:
result = spark.read.format("delta").load(delta_output)

result.orderBy("window.start").show(truncate=False)

+------------------------------------------+--------+-----+
|window                                    |customer|count|
+------------------------------------------+--------+-----+
|{2026-10-02 10:00:00, 2026-10-02 10:05:00}|Rahul   |1    |
|{2026-10-02 10:05:00, 2026-10-02 10:10:00}|Rahul   |1    |
|{2026-10-02 10:15:00, 2026-10-02 10:20:00}|Rahul   |1    |
|{2026-10-02 10:20:00, 2026-10-02 10:25:00}|Rahul   |1    |
+------------------------------------------+--------+-----+



In [0]:
spark.read.format("delta").load(delta_output).createOrReplaceTempView("delta_output")

In [0]:
spark.sql("SELECT * FROM delta_output").show(truncate=False)

+------------------------------------------+--------+-----+
|window                                    |customer|count|
+------------------------------------------+--------+-----+
|{2026-10-02 10:15:00, 2026-10-02 10:20:00}|Rahul   |1    |
|{2026-10-02 10:05:00, 2026-10-02 10:10:00}|Rahul   |1    |
|{2026-10-02 10:00:00, 2026-10-02 10:05:00}|Rahul   |1    |
|{2026-10-02 10:20:00, 2026-10-02 10:25:00}|Rahul   |1    |
+------------------------------------------+--------+-----+

